# Food Image Classification — 80 Classes
### Transfer Learning with EfficientNetB0 & ResNet50 on Kaggle GPU

**Dataset:** 80 classes × 50 images = ~4,000 images  
**Strategy:** Heavy augmentation + frozen base → gradual unfreezing (fine-tuning)  
**GPU:** Kaggle P100 / T4

## 1. Install & Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    resnet50, ResNet50_Weights
)

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# ============================================================
# CONFIG — Modify these to fit your dataset path and needs
# ============================================================
class CFG:
    # --- Paths ---
    DATA_DIR     = '/kaggle/input/your-dataset-name'  # <-- CHANGE THIS
    TRAIN_DIR    = f'{DATA_DIR}/train'
    VAL_DIR      = f'{DATA_DIR}/val'        # If no val folder, set to None
    TEST_DIR     = f'{DATA_DIR}/test'       # Optional
    OUTPUT_DIR   = '/kaggle/working'

    # --- Model ---
    MODEL_NAME   = 'efficientnet_b0'  # 'efficientnet_b0' or 'resnet50'
    NUM_CLASSES  = 80
    IMG_SIZE     = 224

    # --- Training ---
    BATCH_SIZE   = 32
    NUM_WORKERS  = 4
    PIN_MEMORY   = True

    # Phase 1: Train only classifier head (frozen backbone)
    PHASE1_EPOCHS = 10
    PHASE1_LR     = 1e-3

    # Phase 2: Fine-tune entire network
    PHASE2_EPOCHS = 20
    PHASE2_LR     = 1e-4

    # --- Augmentation ---
    MIXUP_ALPHA  = 0.2   # Set 0 to disable
    LABEL_SMOOTH = 0.1

    # --- Val split (used if VAL_DIR doesn't exist) ---
    VAL_SPLIT    = 0.2

print('Config loaded ✅')
os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

## 3. Dataset Exploration

In [ ]:
# Explore dataset structure
train_path = Path(CFG.TRAIN_DIR)
classes = sorted([d.name for d in train_path.iterdir() if d.is_dir()])
print(f'Total classes found: {len(classes)}')
print(f'First 10 classes: {classes[:10]}')

# Count images per class
class_counts = {}
for cls in classes:
    imgs = list((train_path / cls).glob('*'))
    imgs = [f for f in imgs if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']]
    class_counts[cls] = len(imgs)

counts = list(class_counts.values())
print(f'\nImages per class — Min: {min(counts)}, Max: {max(counts)}, Mean: {np.mean(counts):.1f}')
print(f'Total training images: {sum(counts)}')

# Plot class distribution
plt.figure(figsize=(20, 4))
plt.bar(range(len(classes)), [class_counts[c] for c in classes])
plt.xticks(range(len(classes)), classes, rotation=90, fontsize=7)
plt.title('Images per Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize sample images from random classes
sample_classes = random.sample(classes, min(8, len(classes)))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, cls in enumerate(sample_classes):
    cls_path = train_path / cls
    img_files = list(cls_path.glob('*'))
    img_files = [f for f in img_files if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    if img_files:
        img = mpimg.imread(str(random.choice(img_files)))
        axes[i].imshow(img)
        axes[i].set_title(cls, fontsize=10)
        axes[i].axis('off')
plt.suptitle('Sample Food Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Data Transforms & Loaders

In [ ]:
# ImageNet normalization stats (used by both EfficientNet and ResNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE + 32, CFG.IMG_SIZE + 32)),
    transforms.RandomCrop(CFG.IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2)),  # Cutout-style
])

val_transforms = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# TTA (Test-Time Augmentation) transforms
tta_transforms = [
    val_transforms,
    transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE + 20, CFG.IMG_SIZE + 20)),
        transforms.CenterCrop(CFG.IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]),
]

print('Transforms defined ✅')

In [ ]:
from torch.utils.data import random_split

# --- Load datasets ---
val_dir_exists = CFG.VAL_DIR and os.path.isdir(CFG.VAL_DIR)

if val_dir_exists:
    print('Found separate val directory — using it directly.')
    train_dataset = datasets.ImageFolder(CFG.TRAIN_DIR, transform=train_transforms)
    val_dataset   = datasets.ImageFolder(CFG.VAL_DIR,   transform=val_transforms)
else:
    print(f'No val dir found — splitting train {int((1-CFG.VAL_SPLIT)*100)}/{int(CFG.VAL_SPLIT*100)}')
    full_dataset = datasets.ImageFolder(CFG.TRAIN_DIR, transform=train_transforms)
    n_val   = int(len(full_dataset) * CFG.VAL_SPLIT)
    n_train = len(full_dataset) - n_val
    train_dataset, val_dataset = random_split(
        full_dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
    # Apply correct transforms to val split
    val_dataset.dataset = datasets.ImageFolder(CFG.TRAIN_DIR, transform=val_transforms)

CLASS_NAMES = (
    train_dataset.classes
    if hasattr(train_dataset, 'classes')
    else train_dataset.dataset.classes
)
print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')
print(f'Classes       : {len(CLASS_NAMES)}')

# --- Weighted sampler for class imbalance ---
if hasattr(train_dataset, 'targets'):
    targets = train_dataset.targets
else:
    targets = [train_dataset.dataset.targets[i] for i in train_dataset.indices]

class_sample_count = np.bincount(targets)
weights = 1.0 / class_sample_count
sample_weights = torch.tensor([weights[t] for t in targets], dtype=torch.float)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=CFG.BATCH_SIZE, sampler=sampler,
    num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
    num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY
)

print('DataLoaders ready ✅')

## 5. Model Definition

In [ ]:
def build_model(model_name: str, num_classes: int, freeze_backbone: bool = True):
    """
    Build EfficientNetB0 or ResNet50 with a custom classifier head.
    freeze_backbone=True for Phase 1 (train head only).
    """
    if model_name == 'efficientnet_b0':
        model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features

        # Custom head with dropout
        model.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )
        backbone_params = list(model.features.parameters())

    elif model_name == 'resnet50':
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        in_features = model.fc.in_features

        model.fc = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(p=0.3),
            nn.Linear(512, num_classes)
        )
        # All layers except fc
        backbone_params = [
            p for name, p in model.named_parameters()
            if not name.startswith('fc')
        ]
    else:
        raise ValueError(f'Unknown model: {model_name}')

    if freeze_backbone:
        for p in backbone_params:
            p.requires_grad = False
        print(f'Backbone frozen. Only training head.')
    else:
        for p in backbone_params:
            p.requires_grad = True
        print(f'All layers unfrozen for fine-tuning.')

    total_params     = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model: {model_name}')
    print(f'Total params    : {total_params:,}')
    print(f'Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)')

    return model.to(DEVICE)


# --- Mixup Augmentation ---
def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('Model builder ready ✅')

## 6. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, use_mixup=True):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, leave=False, desc='Train'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if use_mixup and CFG.MIXUP_ALPHA > 0:
            images, labels_a, labels_b, lam = mixup_data(images, labels, CFG.MIXUP_ALPHA)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            if use_mixup and CFG.MIXUP_ALPHA > 0:
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, leave=False, desc='Val'):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels


def run_training(model, train_loader, val_loader, epochs, lr,
                 phase_name='Phase', use_mixup=True, scheduler_type='cosine'):
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTH)

    # Separate LR for backbone vs head (useful in Phase 2)
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(head_params, lr=lr, weight_decay=1e-4)

    if scheduler_type == 'onecycle':
        scheduler = OneCycleLR(
            optimizer, max_lr=lr,
            steps_per_epoch=len(train_loader), epochs=epochs,
            pct_start=0.3, div_factor=10, final_div_factor=100
        )
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)

    scaler = torch.cuda.amp.GradScaler()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_model_path = f'{CFG.OUTPUT_DIR}/best_{CFG.MODEL_NAME}.pth'

    print(f'\n{"="*55}')
    print(f'  {phase_name} — {epochs} epochs, LR={lr}')
    print(f'{"="*55}')

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, use_mixup
        )
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

        if scheduler_type == 'cosine':
            scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        marker = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            marker = '  ← best'

        print(f'Epoch {epoch:03d}/{epochs} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}{marker}')

    print(f'\nBest Val Accuracy: {best_val_acc:.4f}')
    return history, best_model_path


print('Training functions ready ✅')

## 7. Phase 1 — Train Classifier Head (Frozen Backbone)

In [ ]:
# Build model with frozen backbone
model = build_model(CFG.MODEL_NAME, CFG.NUM_CLASSES, freeze_backbone=True)

history_p1, best_model_path = run_training(
    model, train_loader, val_loader,
    epochs=CFG.PHASE1_EPOCHS,
    lr=CFG.PHASE1_LR,
    phase_name='Phase 1 – Head Only',
    use_mixup=False,       # No mixup in phase 1 (model hasn't warmed up)
    scheduler_type='cosine'
)

## 8. Phase 2 — Fine-Tune Full Network

In [ ]:
# Load best weights from Phase 1, then unfreeze
model.load_state_dict(torch.load(best_model_path))
model = build_model(CFG.MODEL_NAME, CFG.NUM_CLASSES, freeze_backbone=False)
model.load_state_dict(torch.load(best_model_path))

# For fine-tuning: use differential learning rates
# Backbone gets 10x lower LR than head
if CFG.MODEL_NAME == 'efficientnet_b0':
    backbone_params = list(model.features.parameters())
    head_params     = list(model.classifier.parameters())
else:  # resnet50
    head_params     = list(model.fc.parameters())
    backbone_params = [p for p in model.parameters() if p not in set(head_params)]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CFG.PHASE2_LR / 10},
    {'params': head_params,     'lr': CFG.PHASE2_LR}
], weight_decay=1e-4)

criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTH)
scheduler = CosineAnnealingLR(optimizer, T_max=CFG.PHASE2_EPOCHS, eta_min=1e-7)
scaler    = torch.cuda.amp.GradScaler()

history_p2 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

print(f'\n{"="*55}')
print(f'  Phase 2 – Full Fine-Tune ({CFG.PHASE2_EPOCHS} epochs)')
print(f'{"="*55}')

for epoch in range(1, CFG.PHASE2_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, use_mixup=True
    )
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    history_p2['train_loss'].append(train_loss)
    history_p2['train_acc'].append(train_acc)
    history_p2['val_loss'].append(val_loss)
    history_p2['val_acc'].append(val_acc)

    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        marker = '  ← best'

    print(f'Epoch {epoch:03d}/{CFG.PHASE2_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}{marker}')

print(f'\nBest Val Accuracy (Phase 2): {best_val_acc:.4f}')

## 9. Training Curves

In [ ]:
def plot_history(h1, h2=None):
    # Combine histories
    if h2:
        combined = {k: h1[k] + h2[k] for k in h1}
        phase1_end = len(h1['train_loss'])
    else:
        combined = h1
        phase1_end = None

    epochs_range = range(1, len(combined['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs_range, combined['train_loss'], label='Train Loss', color='royalblue')
    axes[0].plot(epochs_range, combined['val_loss'],   label='Val Loss',   color='tomato')
    if phase1_end:
        axes[0].axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune start')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

    axes[1].plot(epochs_range, [a*100 for a in combined['train_acc']], label='Train Acc', color='royalblue')
    axes[1].plot(epochs_range, [a*100 for a in combined['val_acc']],   label='Val Acc',   color='tomato')
    if phase1_end:
        axes[1].axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune start')
    axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].set_xlabel('Epoch')

    plt.suptitle(f'{CFG.MODEL_NAME} Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{CFG.OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history_p1, history_p2)

## 10. Evaluation — Classification Report & Confusion Matrix

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path))

criterion = nn.CrossEntropyLoss()
val_loss, val_acc, all_preds, all_labels = evaluate(model, val_loader, criterion)

print(f'Final Val Accuracy: {val_acc*100:.2f}%')
print(f'Final Val Loss    : {val_loss:.4f}')
print()
print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=3))

In [ ]:
# Confusion matrix (normalized)
cm = confusion_matrix(all_labels, all_preds, normalize='true')

fig_size = max(16, len(CLASS_NAMES) // 3)
plt.figure(figsize=(fig_size, fig_size))
sns.heatmap(
    cm, annot=(len(CLASS_NAMES) <= 30), fmt='.2f',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cmap='Blues', linewidths=0.5
)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('True', fontsize=12)
plt.title('Normalized Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(rotation=0,  fontsize=7)
plt.tight_layout()
plt.savefig(f'{CFG.OUTPUT_DIR}/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class accuracy — find hardest / easiest classes
per_class_acc = cm.diagonal()
df_acc = pd.DataFrame({'class': CLASS_NAMES, 'accuracy': per_class_acc})
df_acc = df_acc.sort_values('accuracy')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hardest 15
axes[0].barh(df_acc['class'].head(15), df_acc['accuracy'].head(15), color='tomato')
axes[0].set_title('15 Hardest Classes'); axes[0].set_xlabel('Accuracy')
axes[0].axvline(x=val_acc, color='gray', linestyle='--', label='Mean')
axes[0].legend()

# Easiest 15
axes[1].barh(df_acc['class'].tail(15), df_acc['accuracy'].tail(15), color='seagreen')
axes[1].set_title('15 Easiest Classes'); axes[1].set_xlabel('Accuracy')

plt.suptitle('Per-Class Accuracy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Test-Time Augmentation (TTA) Inference

In [ ]:
@torch.no_grad()
def predict_with_tta(model, image_paths, tta_transforms_list, class_names):
    """
    Predict single images with TTA. Averages softmax probabilities over augmentations.
    """
    from PIL import Image
    model.eval()
    results = []

    for img_path in image_paths:
        img = Image.open(img_path).convert('RGB')
        probs_avg = torch.zeros(len(class_names)).to(DEVICE)

        for tfm in tta_transforms_list:
            tensor = tfm(img).unsqueeze(0).to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(tensor)
            probs_avg += torch.softmax(logits.squeeze(), dim=0)

        probs_avg /= len(tta_transforms_list)
        pred_idx  = probs_avg.argmax().item()
        results.append({
            'path'      : img_path,
            'predicted' : class_names[pred_idx],
            'confidence': probs_avg[pred_idx].item()
        })

    return pd.DataFrame(results)


# ---- TTA on val set (batch version) ----
@torch.no_grad()
def tta_evaluate(model, dataset_path, tta_transforms_list, class_names, batch_size=32):
    from PIL import Image
    model.eval()
    all_probs, all_labels_list = [], []

    base_dataset = datasets.ImageFolder(dataset_path, transform=tta_transforms_list[0])
    loader = DataLoader(base_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    for tfm_idx, tfm in enumerate(tta_transforms_list):
        tfm_dataset = datasets.ImageFolder(dataset_path, transform=tfm)
        tfm_loader  = DataLoader(tfm_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
        epoch_probs = []
        labels_captured = (tfm_idx == 0)

        for images, labels in tqdm(tfm_loader, desc=f'TTA aug {tfm_idx+1}/{len(tta_transforms_list)}', leave=False):
            images = images.to(DEVICE)
            with torch.cuda.amp.autocast():
                probs = torch.softmax(model(images), dim=1)
            epoch_probs.append(probs.cpu())
            if labels_captured:
                all_labels_list.extend(labels.numpy())

        all_probs.append(torch.cat(epoch_probs, dim=0))

    avg_probs = torch.stack(all_probs, dim=0).mean(dim=0)  # [N, C]
    preds = avg_probs.argmax(dim=1).numpy()
    labels_np = np.array(all_labels_list)
    acc = (preds == labels_np).mean()
    return acc, preds, labels_np


# Run TTA on val set
val_dir_for_tta = CFG.VAL_DIR if val_dir_exists else CFG.TRAIN_DIR
tta_acc, tta_preds, tta_labels = tta_evaluate(
    model, val_dir_for_tta, tta_transforms, CLASS_NAMES
)
print(f'TTA Val Accuracy: {tta_acc*100:.2f}%')

## 12. Save & Export

In [ ]:
# Save final checkpoint with metadata
final_ckpt = {
    'model_name'    : CFG.MODEL_NAME,
    'num_classes'   : CFG.NUM_CLASSES,
    'class_names'   : CLASS_NAMES,
    'img_size'      : CFG.IMG_SIZE,
    'state_dict'    : model.state_dict(),
    'val_accuracy'  : best_val_acc,
    'tta_accuracy'  : tta_acc,
}
torch.save(final_ckpt, f'{CFG.OUTPUT_DIR}/food_classifier_final.pth')
print(f'Model saved to {CFG.OUTPUT_DIR}/food_classifier_final.pth')

# Save class list
with open(f'{CFG.OUTPUT_DIR}/class_names.txt', 'w') as f:
    f.write('\n'.join(CLASS_NAMES))
print(f'Class names saved.')

# Save per-class accuracy CSV
df_acc.to_csv(f'{CFG.OUTPUT_DIR}/per_class_accuracy.csv', index=False)
print(f'Per-class accuracy saved.')

## 13. Inference — Single Image

In [ ]:
from PIL import Image

def predict_image(img_path, model, class_names, transform, top_k=5):
    """Predict the top-k classes for a single image."""
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad(), torch.cuda.amp.autocast():
        probs = torch.softmax(model(tensor), dim=1).squeeze().cpu()

    top_probs, top_idxs = probs.topk(top_k)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img); axes[0].axis('off'); axes[0].set_title('Input Image')
    colors = ['royalblue'] + ['lightgray'] * (top_k - 1)
    axes[1].barh(
        [class_names[i] for i in top_idxs.numpy()[::-1]],
        top_probs.numpy()[::-1],
        color=colors[::-1]
    )
    axes[1].set_xlim(0, 1); axes[1].set_xlabel('Probability')
    axes[1].set_title(f'Top-{top_k} Predictions')
    plt.tight_layout()
    plt.show()

    print(f'\nTop prediction: {class_names[top_idxs[0]]} ({top_probs[0]*100:.1f}%)')


# Usage:
# predict_image('/path/to/your/image.jpg', model, CLASS_NAMES, val_transforms)